In [1]:
#!/usr/bin/env python3
"""
================================================================================
MiPa: Mixed Patch Visible-Infrared Modality Agnostic Object Detection
Reproduction & Controlled Experimental Study on LLVIP Dataset
================================================================================
Paper: Medeiros et al., "Mixed Patch Visible-Infrared Modality Agnostic Object
       Detection", WACV 2025.
       https://openaccess.thecvf.com/content/WACV2025/papers/Medeiros_Mixed_Patch_
       Visible-Infrared_Modality_Agnostic_Object_Detection_WACV_2025_paper.pdf

Target environment: Kaggle Free-Tier (NVIDIA T4, 16 GB VRAM)
Dataset: LLVIP (pedestrian detection, paired visible/infrared)

HOW TO RUN ON KAGGLE:
  1. Upload the LLVIP dataset as a Kaggle dataset. Expected structure:
     /kaggle/input/llvip/
       ├── visible/  (or infrared/)
       │   ├── train/  *.jpg
       │   └── test/   *.jpg
       ├── infrared/
       │   ├── train/  *.jpg
       │   └── test/   *.jpg
       └── Annotations/  *.xml  (VOC format)
  2. Create a new Kaggle Notebook, enable GPU (T4), paste this entire file.
  3. Run. All outputs go to /kaggle/working/mipa_results/

EXPERIMENTS:
  Exp1: IR-only baseline (unimodal upper bound for IR)
  Exp2: Both naive ρ=0.50 (simple dataset blend)
  Exp3: MiPa Variable ρ ~ U(0,1) (core method)
  Exp4: MiPa + MA γ=0.10 (modality agnostic module)
  Exp5: MiPa Fixed ρ=0.25 (fixed ratio ablation)
  Exp6: MiPa Variable ρ + Frozen backbone (fine-tune head only)

LIMITATIONS:
  - Uses Faster R-CNN + ResNet-50-FPN instead of DINO + SWIN (memory).
    The paper's core MiPa idea is backbone-agnostic; we test it on CNN.
  - Trains on a subset (MAX_TRAIN_SAMPLES) for time constraints.
  - Evaluates on a subset (MAX_VAL_SAMPLES) of test set.
  - Fewer epochs than the paper (6 vs 12).
  - Lower resolution (640x512 → 512x416) to save VRAM.
  - MiPa patch mixing is done at the image-pixel level (grid of patches)
    rather than at ViT token level, since we use a CNN backbone.
    This is a faithful adaptation: the paper patchifies then mixes;
    we mix patches in pixel space, then feed to the CNN.
================================================================================
"""


# KGAT_ea1143c44282fe00a670b1b05c62112a
# export KAGGLE_API_TOKEN=KGAT_ea1143c44282fe00a670b1b05c62112a
# kaggle competitions list
import os

import sys
import gc
import json
import time
import math
import random
import warnings
import datetime
import traceback
# code to run in collab----
# !pip install kaggle
# !kaggle datasets download -d riyajain1040/llvip-dataset
# !ls /content/llvip-dataset/
# !unzip /content/llvip-dataset/llvip-dataset.zip -d /content/llvip-dataset


In [2]:

import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

import torchvision
from torchvision import transforms as T
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,
    FasterRCNN_ResNet50_FPN_V2_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import box_iou

from PIL import Image


In [19]:

warnings.filterwarnings("ignore", category=UserWarning)

# ============================================================================
# SECTION 1: CONFIGURATION
# ============================================================================
@dataclass
class Config:
    """Central configuration for all experiments."""
 
    # ----- Paths -----
    dataset_root: str = ""
    output_dir: str = "/kaggle/working/mipa_results"
 
    # ----- Dataset -----
    max_train_samples: int = 4000
    max_val_samples: int = 800
    img_height: int = 512
    img_width: int = 640
    num_classes: int = 2
    class_names: tuple = ("background", "pedestrian")
 
    # ----- MiPa Patch Mixing -----
    patch_size: int = 32
    rho_mode: str = "variable"          # ONE declaration only. Valid values:
                                         # "variable", "fixed", "beta_centered",
                                         # "beta_bimodal", "truncated_uniform", "gaussian"
    rho_fixed_value: float = 0.50
    beta_a: float = 2.0                 # Alpha param for Beta distribution
    beta_b: float = 2.0                 # Beta param for Beta distribution
 
    # ----- Modality Agnostic (MA) Module -----
    use_ma_module: bool = True           # Default ON for your study
    ma_gamma: float = 0.10
    ma_feature_layer: str = "0"
 
    # ----- Training -----
    backbone: str = "resnet50"
    freeze_backbone: bool = False
    num_epochs: int = 6
    batch_size: int = 16
    grad_accum_steps: int = 1            # Effective batch = 16
    lr: float = 1e-4
    weight_decay: float = 1e-4
    lr_scheduler: str = "cosine"
    use_amp: bool = True
    seed: int = 42
 
    # ----- Evaluation -----
    iou_thresholds: tuple = (0.50,)
    eval_every_n_epochs: int = 2
    save_predictions: bool = True
    num_vis_samples: int = 8
 
    # ----- Experiment Control -----
    experiment_name: str = "default"
    modality_train: str = "mipa"

from pathlib import Path

# Define the root based on your unzip location

# def verify_llvip_structure(root):
#     print(f"Checking LLVIP at: {root}\n")
    
#     paths_to_check = [
#         "visible/train",
#         "visible/test",
#         "infrared/train",
#         "infrared/test",
#         "Annotations"
#     ]
    
#     all_ok = True
#     for rel_path in paths_to_check:
#         full_path = os.path.join(root, rel_path)
#         if os.path.exists(full_path):
#             file_count = len([f for f in os.listdir(full_path) if os.path.isfile(os.path.join(full_path, f))])
#             print(f"[OK] {rel_path:<15} | Found {file_count} files")
#         else:
#             print(f"[MISSING] {rel_path:<15} | Path not found!")
#             all_ok = False
            
#     if all_ok:
#         print("\nStructure looks correct! You can now use this path in your main Config.")
#     else:
#         print("\nSome paths are missing. Please check the unzip output or folder names.")

# verify_llvip_structure(dataset_root)
# def auto_detect_dataset() -> str:
#     """Auto-detect LLVIP dataset root path."""
#     candidates = [
#         "/kaggle/input/llvip",
#         "/kaggle/input/llvip-dataset",
#         "/kaggle/input/llvip-dataset/LLVIP",
#         "/kaggle/input/LLVIP",
#         "/kaggle/input",
#         "./LLVIP",
#         "./data/LLVIP",
#     ]
#     for c in candidates:
#         if os.path.isdir(c):
#             # Check for expected subdirectories
#             has_vis = os.path.isdir(os.path.join(c, "visible"))
#             has_ir = os.path.isdir(os.path.join(c, "infrared"))
#             if has_vis and has_ir:
#                 return c
#     # Try one level deeper
#     for c in candidates:
#         if os.path.isdir(c):
#             for sub in os.listdir(c):
#                 sub_path = os.path.join(c, sub)
#                 if os.path.isdir(sub_path):
#                     has_vis = os.path.isdir(os.path.join(sub_path, "visible"))
#                     has_ir = os.path.isdir(os.path.join(sub_path, "infrared"))
#                     if has_vis and has_ir:
#                         return sub_path
#     return ""


def set_seed(seed: int):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True


# ============================================================================
# SECTION 2: DATASET LOADING & ANNOTATION PARSING
# ============================================================================

def parse_voc_xml(xml_path: str) -> List[Dict]:
    """Parse a VOC XML annotation file and return list of bounding boxes."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    objects = []
    for obj in root.findall("object"):
        name = obj.find("name").text.strip().lower()
        bbox = obj.find("bndbox")
        xmin = float(bbox.find("xmin").text)
        ymin = float(bbox.find("ymin").text)
        xmax = float(bbox.find("xmax").text)
        ymax = float(bbox.find("ymax").text)
        # Ensure valid bbox
        if xmax > xmin and ymax > ymin:
            objects.append({
                "class": name,
                "bbox": [xmin, ymin, xmax, ymax],
            })
    return objects


def find_annotation_dirs(dataset_root: str) -> List[str]:
    """Find annotation directory paths."""
    candidates = [
        os.path.join(dataset_root, "Annotations"),
        os.path.join(dataset_root, "annotations"),
        os.path.join(dataset_root, "Annotation"),
        os.path.join(dataset_root, "labels"),
    ]
    for c in candidates:
        if os.path.isdir(c):
            return c
    # Check for annotations inside infrared/train etc.
    return ""


class LLVIPDataset(Dataset):
    """
    LLVIP paired visible/infrared dataset for object detection.

    Each sample returns (image, target) where image is a tensor and target
    is a dict with 'boxes', 'labels', 'image_id'.

    For MiPa training: returns BOTH visible and infrared images so the
    training loop can perform patch mixing.
    """

    def __init__(
        self,
        dataset_root: str,
        split: str = "train",
        modality: str = "both",     # "rgb", "ir", "both"
        max_samples: int = -1,
        img_size: Tuple[int, int] = (416, 512),  # (H, W)
        augment: bool = False,
    ):
        self.dataset_root = dataset_root
        self.split = split
        self.modality = modality
        self.img_size = img_size
        self.augment = augment

        # Build paths
        vis_dir = os.path.join(dataset_root, "visible", split)
        ir_dir = os.path.join(dataset_root, "infrared", split)
        ann_dir = find_annotation_dirs(dataset_root)

        if not os.path.isdir(vis_dir):
            raise FileNotFoundError(
                f"Visible images not found at {vis_dir}\n"
                f"Please ensure LLVIP dataset is at: {dataset_root}\n"
                f"Expected structure: visible/train/, visible/test/, "
                f"infrared/train/, infrared/test/, Annotations/"
            )
        if not os.path.isdir(ir_dir):
            raise FileNotFoundError(f"Infrared images not found at {ir_dir}")

        # Collect image filenames (paired)
        vis_files = sorted([f for f in os.listdir(vis_dir)
                           if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        ir_files = sorted([f for f in os.listdir(ir_dir)
                          if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

        # Find common filenames (paired images)
        vis_stems = {os.path.splitext(f)[0]: f for f in vis_files}
        ir_stems = {os.path.splitext(f)[0]: f for f in ir_files}
        common_stems = sorted(set(vis_stems.keys()) & set(ir_stems.keys()))

        if len(common_stems) == 0:
            raise ValueError(
                f"No paired images found. Visible: {len(vis_files)}, "
                f"IR: {len(ir_files)}. Check filename matching."
            )

        # Limit samples
        if max_samples > 0 and max_samples < len(common_stems):
            common_stems = common_stems[:max_samples]

        self.samples = []
        skipped = 0
        for stem in common_stems:
            vis_path = os.path.join(vis_dir, vis_stems[stem])
            ir_path = os.path.join(ir_dir, ir_stems[stem])

            # Try to find annotation
            ann_path = None
            if ann_dir:
                for ext in [".xml", ".XML"]:
                    candidate = os.path.join(ann_dir, stem + ext)
                    if os.path.isfile(candidate):
                        ann_path = candidate
                        break

            if ann_path is None:
                # Try YOLO-style txt annotations
                yolo_candidates = [
                    os.path.join(dataset_root, "labels", split, stem + ".txt"),
                    os.path.join(dataset_root, "labels", stem + ".txt"),
                ]
                for yc in yolo_candidates:
                    if os.path.isfile(yc):
                        ann_path = yc
                        break

            if ann_path is None:
                skipped += 1
                continue

            self.samples.append({
                "stem": stem,
                "vis_path": vis_path,
                "ir_path": ir_path,
                "ann_path": ann_path,
            })

        if skipped > 0:
            print(f"  [Dataset] Skipped {skipped} images with no annotations")
        print(f"  [Dataset] Loaded {len(self.samples)} paired samples "
              f"({split}, modality={modality})")

    def __len__(self):
        return len(self.samples)

    def _load_and_resize(self, path: str) -> torch.Tensor:
        """Load image, resize, convert to tensor [C,H,W] float32 in [0,1]."""
        img = Image.open(path).convert("RGB")
        img = img.resize((self.img_size[1], self.img_size[0]), Image.BILINEAR)
        img = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
        return img

    def _parse_annotations(self, ann_path: str, orig_w: int, orig_h: int):
        """Parse annotations and scale bounding boxes to target size."""
        boxes = []
        labels = []

        target_h, target_w = self.img_size
        sx = target_w / orig_w
        sy = target_h / orig_h

        if ann_path.endswith(".xml"):
            objects = parse_voc_xml(ann_path)
            for obj in objects:
                x1, y1, x2, y2 = obj["bbox"]
                x1 = max(0, x1 * sx)
                y1 = max(0, y1 * sy)
                x2 = min(target_w, x2 * sx)
                y2 = min(target_h, y2 * sy)
                if x2 > x1 + 1 and y2 > y1 + 1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(1)  # pedestrian = class 1

        elif ann_path.endswith(".txt"):
            # YOLO format: class cx cy w h (normalized)
            with open(ann_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls = int(parts[0])
                        cx, cy, w, h = map(float, parts[1:5])
                        x1 = (cx - w / 2) * target_w
                        y1 = (cy - h / 2) * target_h
                        x2 = (cx + w / 2) * target_w
                        y2 = (cy + h / 2) * target_h
                        x1 = max(0, x1)
                        y1 = max(0, y1)
                        x2 = min(target_w, x2)
                        y2 = min(target_h, y2)
                        if x2 > x1 + 1 and y2 > y1 + 1:
                            boxes.append([x1, y1, x2, y2])
                            labels.append(1)  # pedestrian

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        return boxes, labels

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Load both modalities
        vis_img = self._load_and_resize(sample["vis_path"])
        ir_img = self._load_and_resize(sample["ir_path"])

        # Get original image size for annotation scaling
        with Image.open(sample["vis_path"]) as pil_img:
            orig_w, orig_h = pil_img.size

        boxes, labels = self._parse_annotations(
            sample["ann_path"], orig_w, orig_h
        )

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
        }

        # Data augmentation (simple horizontal flip)
        if self.augment and random.random() > 0.5:
            vis_img = torch.flip(vis_img, [2])
            ir_img = torch.flip(ir_img, [2])
            if len(boxes) > 0:
                w = self.img_size[1]
                new_boxes = boxes.clone()
                new_boxes[:, 0] = w - boxes[:, 2]
                new_boxes[:, 2] = w - boxes[:, 0]
                target["boxes"] = new_boxes

        return vis_img, ir_img, target


def collate_fn(batch):
    """Custom collate for detection: keeps targets as list of dicts."""
    vis_imgs, ir_imgs, targets = zip(*batch)
    return list(vis_imgs), list(ir_imgs), list(targets)


# ============================================================================
# SECTION 3: MiPa PATCH MIXING MODULE
# ============================================================================

class MiPaPatchMixer:
    """
    Mixed Patch (MiPa) module: creates a mosaic of patches from two modalities.
 
    For each image pair (visible, infrared), divides into a grid of patches
    and stochastically assigns each patch to one modality based on ratio rho.
 
    Vectorized implementation — no Python loops over patches.
    """
 
    def __init__(
        self,
        patch_size: int = 32,
        rho_mode: str = "variable",
        rho_fixed_value: float = 0.50,
        beta_a: float = 2.0,
        beta_b: float = 2.0,
    ):
        self.patch_size = patch_size
        self.rho_mode = rho_mode
        self.rho_fixed_value = rho_fixed_value
        self.beta_a = beta_a
        self.beta_b = beta_b
 
    def _sample_rho(self) -> float:
        """Sample rho based on the configured distribution."""
        if self.rho_mode == "variable":
            return random.random()                              # U(0, 1)
        elif self.rho_mode == "fixed":
            return self.rho_fixed_value
        elif self.rho_mode == "beta_centered":
            return float(np.random.beta(self.beta_a, self.beta_b))  # e.g. Beta(2,2)
        elif self.rho_mode == "beta_bimodal":
            return float(np.random.beta(self.beta_a, self.beta_b))  # e.g. Beta(0.5,0.5)
        elif self.rho_mode == "truncated_uniform":
            return random.uniform(0.15, 0.85)
        elif self.rho_mode == "gaussian":
            return max(0.0, min(1.0, np.random.normal(0.5, 0.15)))
        else:
            return 0.5
 
    def mix(
        self,
        vis_imgs: List[torch.Tensor],
        ir_imgs: List[torch.Tensor],
    ) -> Tuple[List[torch.Tensor], List[torch.Tensor]]:
        """
        Mix patches from visible and infrared images (vectorized).
 
        Args:
            vis_imgs: List of [C, H, W] visible tensors
            ir_imgs:  List of [C, H, W] infrared tensors
 
        Returns:
            mixed_imgs: List of [C, H, W] mixed tensors
            modality_maps: List of [nH, nW] binary maps (0=vis, 1=ir)
        """
        if self.rho_mode == "none":
            raise ValueError("Use 'both' modality_train for no-mixing baseline")
 
        mixed_imgs = []
        modality_maps = []
 
        for vis, ir in zip(vis_imgs, ir_imgs):
            C, H, W = vis.shape
            nH = H // self.patch_size
            nW = W // self.patch_size
 
            # Crop to exact patch grid (handles non-divisible sizes)
            H_crop = nH * self.patch_size
            W_crop = nW * self.patch_size
            vis_c = vis[:, :H_crop, :W_crop]
            ir_c = ir[:, :H_crop, :W_crop]
 
            # Sample rho
            rho = self._sample_rho()
 
            # Create random modality map: 1 = IR, 0 = visible
            total_patches = nH * nW
            num_ir_patches = int(round(rho * total_patches))
 
            # Generate random permutation and pick first num_ir_patches
            perm = torch.randperm(total_patches)
            mod_map_flat = torch.zeros(total_patches, dtype=torch.float32)
            mod_map_flat[perm[:num_ir_patches]] = 1.0
            mod_map = mod_map_flat.reshape(nH, nW)  # [nH, nW]
 
            # Upscale modality map to pixel resolution: [1, 1, nH, nW] -> [1, 1, H_crop, W_crop]
            mask = mod_map.unsqueeze(0).unsqueeze(0)  # [1, 1, nH, nW]
            mask = F.interpolate(mask, size=(H_crop, W_crop), mode='nearest')
            mask = mask.squeeze(0)  # [1, H_crop, W_crop]
 
            # Vectorized mix: mask * IR + (1 - mask) * visible
            mixed = mask * ir_c + (1.0 - mask) * vis_c
 
            mixed_imgs.append(mixed)
            modality_maps.append(mod_map)
 
        return mixed_imgs, modality_maps


# ============================================================================
# SECTION 4: MODEL DEFINITION
# ============================================================================

class GradientReversalFunction(torch.autograd.Function):
    """Gradient Reversal Layer (GRL) for modality agnostic training."""

    @staticmethod
    def forward(ctx, x, lambda_val):
        ctx.lambda_val = lambda_val
        return x.clone()

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_val * grad_output, None


class ModalityAgnosticModule(nn.Module):
    """
    Patch-wise Modality Agnostic (MA) module.

    Takes feature maps from the backbone and predicts a modality map.
    Uses GRL to make features modality-invariant via adversarial training.

    The loss is BCE between predicted and ground-truth modality maps.
    Lambda increases over training via: λ = 2/(1 + exp(-γ·s)) - 1
    """

    def __init__(self, in_channels: int, gamma: float = 0.10):
        super().__init__()
        self.gamma = gamma
        self.global_step = 0

        # Modality classifier: conv layers → per-patch prediction
        self.classifier = nn.Sequential(
            nn.Conv2d(in_channels, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, kernel_size=1),  # Per-spatial-location prediction
        )

    def get_lambda(self) -> float:
        """Compute lambda based on current training step."""
        s = self.global_step
        lam = 2.0 / (1.0 + math.exp(-self.gamma * s)) - 1.0
        return lam

    def forward(
        self,
        features: torch.Tensor,
        modality_maps: Optional[List[torch.Tensor]] = None,
    ) -> Optional[torch.Tensor]:
        """
        Args:
            features: [B, C, H_feat, W_feat] from backbone
            modality_maps: List of [nH, nW] ground-truth modality maps

        Returns:
            ma_loss: scalar loss (or None if modality_maps not provided)
        """
        if modality_maps is None:
            return None

        lam = self.get_lambda()

        # Apply GRL
        reversed_features = GradientReversalFunction.apply(features, lam)

        # Predict modality map
        pred = self.classifier(reversed_features)  # [B, 1, H_f, W_f]
        pred = pred.squeeze(1)  # [B, H_f, W_f]

        # Resize ground-truth modality maps to match feature map size
        B, H_f, W_f = pred.shape
        targets = []
        for mm in modality_maps:
            mm_resized = F.interpolate(
                mm.unsqueeze(0).unsqueeze(0).to(pred.device),
                size=(H_f, W_f),
                mode="nearest",
            ).squeeze(0).squeeze(0)
            targets.append(mm_resized)
        target_maps = torch.stack(targets, dim=0)  # [B, H_f, W_f]

        # BCE loss
        ma_loss = F.binary_cross_entropy_with_logits(pred, target_maps)

        return lam * ma_loss


class MiPaDetector(nn.Module):
    """
    Object detector with MiPa support.

    Wraps torchvision Faster R-CNN with:
    - Optional Modality Agnostic module
    - Feature extraction hooks for MA module
    """

    def __init__(self, config: Config):
        super().__init__()
        self.config = config

        # Build Faster R-CNN with pre-trained backbone
        weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
        self.detector = fasterrcnn_resnet50_fpn_v2(weights=weights)

        # Replace head for single-class detection (pedestrian)
        in_features = self.detector.roi_heads.box_predictor.cls_score.in_features
        self.detector.roi_heads.box_predictor = FastRCNNPredictor(
            in_features, config.num_classes
        )

        # Freeze backbone if requested
        if config.freeze_backbone:
            for param in self.detector.backbone.parameters():
                param.requires_grad = False
            print("  [Model] Backbone frozen")

        # Modality Agnostic module
        self.ma_module = None
        if config.use_ma_module:
            # ResNet50-FPN feature channels at different levels
            # FPN output channels = 256
            self.ma_module = ModalityAgnosticModule(
                in_channels=256,
                gamma=config.ma_gamma,
            )
            print(f"  [Model] MA module enabled (γ={config.ma_gamma})")

        # Hook to capture intermediate features for MA module
        self._features_for_ma = None
        if self.ma_module is not None:
            # Register hook on FPN output
            def hook_fn(module, input, output):
                # output is an OrderedDict of feature maps
                # Take the first level (highest resolution)
                if isinstance(output, dict):
                    key = list(output.keys())[0]
                    self._features_for_ma = output[key]
                elif isinstance(output, (list, tuple)):
                    self._features_for_ma = output[0]

            self.detector.backbone.register_forward_hook(hook_fn)

        # Count parameters
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  [Model] Parameters: {total:,} total, {trainable:,} trainable")

    def forward(
        self,
        images: List[torch.Tensor],
        targets: Optional[List[Dict]] = None,
        modality_maps: Optional[List[torch.Tensor]] = None,
    ):
        """
        Forward pass.

        In training mode: returns detection losses + optional MA loss.
        In eval mode: returns predictions.
        """
        if self.training and targets is not None:
            # Training: get detection losses
            loss_dict = self.detector(images, targets)

            # Compute MA loss if enabled
            ma_loss = None
            if (self.ma_module is not None
                    and modality_maps is not None
                    and self._features_for_ma is not None):
                ma_loss = self.ma_module(self._features_for_ma, modality_maps)
                if ma_loss is not None:
                    loss_dict["loss_ma"] = ma_loss

            self._features_for_ma = None
            return loss_dict
        else:
            # Evaluation: get predictions
            self._features_for_ma = None
            return self.detector(images)


# ============================================================================
# SECTION 5: EVALUATION METRICS
# ============================================================================

def compute_ap(
    predictions: List[Dict],
    ground_truths: List[Dict],
    iou_threshold: float = 0.50,
    num_classes: int = 2,
) -> Dict[str, float]:
    """
    Compute Average Precision (AP) at given IoU threshold.

    Returns dict with 'AP50', 'precision', 'recall', 'num_gt', 'num_pred'.
    """
    # Collect all predictions and ground truths per class
    all_pred_boxes = []
    all_pred_scores = []
    all_gt_boxes = []
    all_gt_matched = []

    for pred, gt in zip(predictions, ground_truths):
        pred_boxes = pred.get("boxes", torch.tensor([]))
        pred_scores = pred.get("scores", torch.tensor([]))
        pred_labels = pred.get("labels", torch.tensor([]))
        gt_boxes = gt.get("boxes", torch.tensor([]))
        gt_labels = gt.get("labels", torch.tensor([]))

        # Filter for pedestrian class (label=1)
        if len(pred_labels) > 0:
            mask = pred_labels == 1
            pred_boxes = pred_boxes[mask]
            pred_scores = pred_scores[mask]
        if len(gt_labels) > 0:
            mask = gt_labels == 1
            gt_boxes = gt_boxes[mask]

        all_pred_boxes.append(pred_boxes.cpu() if len(pred_boxes) > 0
                             else torch.zeros(0, 4))
        all_pred_scores.append(pred_scores.cpu() if len(pred_scores) > 0
                              else torch.zeros(0))
        all_gt_boxes.append(gt_boxes.cpu() if len(gt_boxes) > 0
                           else torch.zeros(0, 4))
        all_gt_matched.append(torch.zeros(len(gt_boxes), dtype=torch.bool))

    # Total counts
    num_gt = sum(len(g) for g in all_gt_boxes)
    num_pred = sum(len(p) for p in all_pred_boxes)

    if num_gt == 0:
        return {"AP50": 0.0, "precision": 0.0, "recall": 0.0,
                "num_gt": 0, "num_pred": num_pred}

    # Aggregate all predictions with image index
    all_scores = []
    all_tp = []

    for img_idx in range(len(predictions)):
        pred_boxes = all_pred_boxes[img_idx]
        pred_scores = all_pred_scores[img_idx]
        gt_boxes = all_gt_boxes[img_idx]
        gt_matched = all_gt_matched[img_idx]

        if len(pred_boxes) == 0:
            continue

        # Sort by score descending
        sorted_indices = torch.argsort(pred_scores, descending=True)

        for si in sorted_indices:
            score = pred_scores[si].item()
            pred_box = pred_boxes[si].unsqueeze(0)

            tp = 0
            if len(gt_boxes) > 0:
                ious = box_iou(pred_box, gt_boxes).squeeze(0)
                max_iou, max_idx = ious.max(0) if len(ious.shape) > 0 else (ious, torch.tensor(0))
                if max_iou.item() >= iou_threshold and not gt_matched[max_idx]:
                    tp = 1
                    gt_matched[max_idx] = True

            all_scores.append(score)
            all_tp.append(tp)

    if len(all_scores) == 0:
        return {"AP50": 0.0, "precision": 0.0, "recall": 0.0,
                "num_gt": num_gt, "num_pred": 0}

    # Sort by score
    sorted_indices = np.argsort(-np.array(all_scores))
    all_tp = np.array(all_tp)[sorted_indices]

    # Compute precision-recall curve
    cum_tp = np.cumsum(all_tp)
    cum_fp = np.cumsum(1 - all_tp)
    precision = cum_tp / (cum_tp + cum_fp)
    recall = cum_tp / num_gt

    # 11-point interpolation AP
    ap = 0.0
    for t in np.arange(0, 1.1, 0.1):
        prec_at_recall = precision[recall >= t]
        if len(prec_at_recall) > 0:
            ap += prec_at_recall.max() / 11.0

    # Final precision/recall at score threshold 0.5
    mask_05 = np.array(all_scores)[sorted_indices] >= 0.5
    final_prec = precision[mask_05][-1] if mask_05.any() else 0.0
    final_rec = recall[mask_05][-1] if mask_05.any() else 0.0

    return {
        "AP50": round(float(ap) * 100, 2),
        "precision": round(float(final_prec) * 100, 2),
        "recall": round(float(final_rec) * 100, 2),
        "num_gt": num_gt,
        "num_pred": num_pred,
    }


# ============================================================================
# SECTION 6: TRAINING LOOP
# ============================================================================

def train_one_epoch(
    model: MiPaDetector,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
    config: Config,
    epoch: int,
    device: torch.device,
    patch_mixer: Optional[MiPaPatchMixer] = None,
    scheduler: Optional[Any] = None,
) -> Dict[str, float]:
    """Train for one epoch. Returns dict of average losses."""
    model.train()
    loss_accum = defaultdict(float)
    num_batches = 0
    optimizer.zero_grad()

    total_batches = len(dataloader)
    log_interval = max(1, total_batches // 5)
    epoch_start = time.time()

    for batch_idx, (vis_imgs, ir_imgs, targets) in enumerate(dataloader):
        step_start = time.time()

        # Prepare images based on training modality
        modality_maps = None

        if config.modality_train == "ir":
            # IR-only: use infrared images
            images = [img.to(device) for img in ir_imgs]

        elif config.modality_train == "rgb":
            # RGB-only: use visible images
            images = [img.to(device) for img in vis_imgs]

        elif config.modality_train == "both":
            # Both baseline: randomly pick one modality per image
            images = []
            for v, i in zip(vis_imgs, ir_imgs):
                rho = config.rho_fixed_value
                if random.random() < rho:
                    images.append(i.to(device))
                else:
                    images.append(v.to(device))

        elif config.modality_train == "mipa":
            # MiPa: mix patches from both modalities
            if patch_mixer is not None:
                mixed_imgs, mod_maps = patch_mixer.mix(vis_imgs, ir_imgs)
                images = [img.to(device) for img in mixed_imgs]
                modality_maps = mod_maps  # Keep on CPU, moved in MA module
            else:
                images = [img.to(device) for img in vis_imgs]

        else:
            images = [img.to(device) for img in vis_imgs]

        # Move targets to device
        targets_dev = []
        for t in targets:
            td = {}
            for k, v in t.items():
                if isinstance(v, torch.Tensor):
                    td[k] = v.to(device)
                else:
                    td[k] = v
            targets_dev.append(td)

        # Skip batches with no GT boxes (can cause issues)
        valid = any(len(t["boxes"]) > 0 for t in targets_dev)
        if not valid:
            continue

        # Forward pass with mixed precision
        with torch.amp.autocast("cuda", enabled=config.use_amp):
            loss_dict = model(images, targets_dev, modality_maps)
            total_loss = sum(loss_dict.values()) / config.grad_accum_steps

        # Backward pass
        scaler.scale(total_loss).backward()

        # Gradient accumulation step
        # Gradient accumulation step
        if (batch_idx + 1) % config.grad_accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            if scheduler is not None:
                scheduler.step()

        # Accumulate losses for logging
        for k, v in loss_dict.items():
            loss_accum[k] += v.item()
        num_batches += 1

        # Progress logging
        if (batch_idx + 1) % log_interval == 0 or batch_idx == 0:
            elapsed = time.time() - epoch_start
            avg_loss = sum(loss_accum.values()) / max(num_batches, 1)
            gpu_mem = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0
            print(
                f"    [Epoch {epoch+1}] Batch {batch_idx+1}/{total_batches} | "
                f"Loss: {avg_loss:.4f} | "
                f"Time: {elapsed:.1f}s | "
                f"GPU: {gpu_mem:.2f}GB"
            )

        # Clean up
        del images, targets_dev, loss_dict, total_loss
        

    # Final grad step if needed
    if num_batches % config.grad_accum_steps != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    # Average losses
    avg_losses = {k: v / max(num_batches, 1) for k, v in loss_accum.items()}
    avg_losses["total"] = sum(avg_losses.values())
    avg_losses["epoch_time"] = time.time() - epoch_start
    return avg_losses


@torch.no_grad()
def evaluate(
    model: MiPaDetector,
    dataloader: DataLoader,
    device: torch.device,
    modality: str = "ir",  # Evaluate on single modality
    config: Config = None,
) -> Dict[str, float]:
    """Evaluate model on a single modality. Returns metrics dict."""
    model.eval()
    all_predictions = []
    all_ground_truths = []

    eval_start = time.time()
    for batch_idx, (vis_imgs, ir_imgs, targets) in enumerate(dataloader):
        # Select modality for evaluation
        if modality == "ir":
            images = [img.to(device) for img in ir_imgs]
        elif modality == "rgb":
            images = [img.to(device) for img in vis_imgs]
        else:
            images = [img.to(device) for img in ir_imgs]

        # Forward pass
        with torch.amp.autocast("cuda", enabled=config.use_amp if config else True):
            predictions = model(images)

        for pred in predictions:
            all_predictions.append({
                k: v.cpu() for k, v in pred.items()
            })

        for t in targets:
            all_ground_truths.append({
                k: v.cpu() if isinstance(v, torch.Tensor) else v
                for k, v in t.items()
            })

        del images, predictions
        

    eval_time = time.time() - eval_start
    num_images = len(all_predictions)
    time_per_image = eval_time / max(num_images, 1)

    # Compute AP
    metrics = compute_ap(all_predictions, all_ground_truths, iou_threshold=0.50)
    metrics["eval_time"] = round(eval_time, 2)
    metrics["time_per_image"] = round(time_per_image * 1000, 1)  # ms
    metrics["num_images"] = num_images

    return metrics


# ============================================================================
# SECTION 7: EXPERIMENT RUNNER
# ============================================================================
def resume_from_checkpoint(ckpt_path, model, optimizer, device):
    """Load model weights from a checkpoint. Returns the epoch to resume from."""
    if not os.path.isfile(ckpt_path):
        print(f"  [Resume] Checkpoint not found: {ckpt_path}")
        return 0
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    resume_epoch = ckpt["epoch"]
    print(f"  [Resume] Loaded model weights from epoch {resume_epoch} (optimizer reset)")
    return resume_epoch
def run_experiment(config: Config, device: torch.device) -> Dict[str, Any]:
    """
    Run a single experiment configuration.

    Returns a dict with all results, losses, and metrics.
    """
    print(f"\n{'='*70}")
    print(f"  EXPERIMENT: {config.experiment_name}")
    print(f"  Modality: {config.modality_train} | "
          f"rho_mode: {config.rho_mode} | "
          f"Beta({config.beta_a},{config.beta_b}) | "
          f"MA: {config.use_ma_module} (γ={config.ma_gamma}) | "
          f"Backbone: {config.backbone}")
    print(f"  LR: {config.lr} | BS: {config.batch_size} | "
          f"Epochs: {config.num_epochs} | Seed: {config.seed}")
    print(f"{'='*70}")

    set_seed(config.seed)
    os.makedirs(config.output_dir, exist_ok=True)

    # Create dataset
    print("\n  Loading dataset...")
    train_dataset = LLVIPDataset(
        dataset_root=config.dataset_root,
        split="train",
        modality="both",
        max_samples=config.max_train_samples,
        img_size=(config.img_height, config.img_width),
        augment=True,
    )
    val_dataset = LLVIPDataset(
        dataset_root=config.dataset_root,
        split="test",
        modality="both",
        max_samples=config.max_val_samples,
        img_size=(config.img_height, config.img_width),
        augment=False,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=4,
        collate_fn=collate_fn,
        pin_memory=True,
        drop_last=False,
        persistent_workers=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=2,
        collate_fn=collate_fn,
        pin_memory=True,
        persistent_workers=True,
    )

    # Create model
    print("\n  Building model...")
    model = MiPaDetector(config).to(device)
    # below code to enable when running ViT backbone instead of others for dual gpu usage 
    # class DataParallelWrapper(nn.Module):
    # def __init__(self, detector):
    #     super().__init__()
    #     self.detector = detector
    
    # def forward(self, images, targets=None):
    #     if self.training and targets:
    #         loss_dict = self.detector(images, targets)
    #         return sum(loss_dict.values())  # single tensor — DataParallel can gather this
    #     else:
    #         return self.detector(images)

    # Optimizer
    # Optimizer with parameter groups: backbone (slow), head (fast), MA module (medium)
    backbone_params = []
    head_params = []
    ma_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'ma_module' in name:
            ma_params.append(param)
        elif 'detector.backbone' in name:
            backbone_params.append(param)
        else:
            head_params.append(param)

    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": config.lr * 0.1},     # backbone: 1/10th LR
        {"params": head_params,     "lr": config.lr},            # detection head: full LR
        {"params": ma_params,       "lr": config.lr * 0.5},     # MA classifier: half LR
    ], weight_decay=config.weight_decay)

    print(f"  [Optimizer] Backbone params: {len(backbone_params)}, "
          f"Head params: {len(head_params)}, MA params: {len(ma_params)}")

    # Warmup + Cosine Decay scheduler
    total_steps = len(train_loader) * config.num_epochs
    warmup_steps = len(train_loader)  # 1 epoch of warmup

    def lr_lambda(current_step):
        if current_step < warmup_steps:
            # Linear warmup from 0.1x to 1.0x
            return 0.1 + 0.9 * (current_step / warmup_steps)
        else:
            # Cosine decay from 1.0x to 0.01x
            progress = (current_step - warmup_steps) / max(1, total_steps - warmup_steps)
            return 0.01 + 0.99 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    scaler = GradScaler(enabled=config.use_amp)
    # Patch mixer
    patch_mixer = None
    if config.modality_train == "mipa":
        patch_mixer = MiPaPatchMixer(
            patch_size=config.patch_size,
            rho_mode=config.rho_mode,
            rho_fixed_value=config.rho_fixed_value,
            beta_a=config.beta_a,
            beta_b=config.beta_b,
        )
    # Training history
    history = {
        "train_losses": [],
        "val_metrics_ir": [],
        "val_metrics_rgb": [],
        "epoch_times": [],
    }

    best_avg_ap = 0.0
    best_epoch = -1

    # Resume from checkpoint if available
    resume_epoch = 0
    resume_ckpt = os.path.join(config.output_dir, f"epoch_1_{config.experiment_name}.pth")
    if os.path.isfile(resume_ckpt):
        resume_epoch = resume_from_checkpoint(resume_ckpt, model, optimizer, device)

    print(f"\n  Starting training ({config.num_epochs} epochs, resuming from epoch {resume_epoch})...")
    total_start = time.time()

    for epoch in range(resume_epoch, config.num_epochs):
        print(f"\n  --- Epoch {epoch+1}/{config.num_epochs} ---")

        # Train
        train_losses = train_one_epoch(
            model, train_loader, optimizer, scaler,
            config, epoch, device, patch_mixer, scheduler,
        )
        # scheduler.step() is now called per-batch inside train_one_epoch
        scheduler.step()

        history["train_losses"].append(train_losses)
        history["epoch_times"].append(train_losses.get("epoch_time", 0))

        print(f"    Train Loss: {train_losses['total']:.4f} | "
              f"Time: {train_losses['epoch_time']:.1f}s")

        # Evaluate
        if (epoch + 1) % config.eval_every_n_epochs == 0 or epoch == config.num_epochs - 1:
            print(f"    Evaluating...")

            metrics_ir = evaluate(model, val_loader, device, "ir", config)
            metrics_rgb = evaluate(model, val_loader, device, "rgb", config)

            avg_ap50 = (metrics_ir["AP50"] + metrics_rgb["AP50"]) / 2.0

            history["val_metrics_ir"].append({"epoch": epoch + 1, **metrics_ir})
            history["val_metrics_rgb"].append({"epoch": epoch + 1, **metrics_rgb})

            print(f"    IR  AP50: {metrics_ir['AP50']:.2f}% | "
                  f"Prec: {metrics_ir['precision']:.1f}% | "
                  f"Rec: {metrics_ir['recall']:.1f}%")
            print(f"    RGB AP50: {metrics_rgb['AP50']:.2f}% | "
                  f"Prec: {metrics_rgb['precision']:.1f}% | "
                  f"Rec: {metrics_rgb['recall']:.1f}%")
            print(f"    AVG AP50: {avg_ap50:.2f}%")

            # Save best model
            if avg_ap50 > best_avg_ap:
                best_avg_ap = avg_ap50
                best_epoch = epoch + 1
                ckpt_path = os.path.join(
                    config.output_dir,
                    f"best_{config.experiment_name}.pth"
                )
                torch.save({
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "config": asdict(config),
                    "metrics_ir": metrics_ir,
                    "metrics_rgb": metrics_rgb,
                    "avg_ap50": avg_ap50,
                }, ckpt_path)
                print(f"    ✓ Best model saved (AP50_avg={avg_ap50:.2f}%)")

        # Save checkpoint every epoch
        epoch_ckpt_path = os.path.join(
            config.output_dir,
            f"epoch_{epoch+1}_{config.experiment_name}.pth"
        )
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "config": asdict(config),
            "history": history,
        }, epoch_ckpt_path)

        # GPU memory stats
        if torch.cuda.is_available():
            peak_mem = torch.cuda.max_memory_allocated() / 1e9
            print(f"    Peak GPU Memory: {peak_mem:.2f} GB")
            torch.cuda.reset_peak_memory_stats()

    total_time = time.time() - total_start
    print(f"\n  Training complete in {total_time:.1f}s ({total_time/60:.1f}min)")
    print(f"  Best Avg AP50: {best_avg_ap:.2f}% at epoch {best_epoch}")

    # Final evaluation with best model
    if best_epoch > 0:
        ckpt = torch.load(
            os.path.join(config.output_dir, f"best_{config.experiment_name}.pth"),
            map_location=device, weights_only=False,
        )
        model.load_state_dict(ckpt["model_state_dict"])

    # Final metrics
    final_ir = evaluate(model, val_loader, device, "ir", config)
    final_rgb = evaluate(model, val_loader, device, "rgb", config)
    final_avg = (final_ir["AP50"] + final_rgb["AP50"]) / 2.0

    # Compile results
    # Compile results
    results = {
        "experiment_name": config.experiment_name,
        "config": {
            "modality_train": config.modality_train,
            "rho_mode": config.rho_mode,
            "rho_fixed_value": config.rho_fixed_value,
            "beta_a": config.beta_a,
            "beta_b": config.beta_b,
            "use_ma_module": config.use_ma_module,
            "ma_gamma": config.ma_gamma,
            "freeze_backbone": config.freeze_backbone,
            "backbone": config.backbone,
            "lr": config.lr,
            "batch_size": config.batch_size,
            "num_epochs": config.num_epochs,
            "seed": config.seed,
        },
        "final_metrics": {
            "ir_AP50": final_ir["AP50"],
            "rgb_AP50": final_rgb["AP50"],
            "avg_AP50": round(final_avg, 2),
            "ir_precision": final_ir["precision"],
            "ir_recall": final_ir["recall"],
            "rgb_precision": final_rgb["precision"],
            "rgb_recall": final_rgb["recall"],
            "ir_time_per_image_ms": final_ir["time_per_image"],
            "rgb_time_per_image_ms": final_rgb["time_per_image"],
        },
        "best_epoch": best_epoch,
        "total_time_s": round(total_time, 1),
        "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2)
            if torch.cuda.is_available() else 0,
        "history": history,
    }

    # Clean up
    del model, optimizer, scheduler, scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


# ============================================================================
# SECTION 8: EXPERIMENT DEFINITIONS & MAIN
# ============================================================================

def define_experiments(dataset_root: str, output_dir: str) -> List[Config]:
    """
    Define experiments for critique & improvement study.
 
    Group A: rho distribution comparison (all MiPa + MA, ResNet50 backbone)
    Group B: backbone comparison (all MiPa variable + MA)
    """
    base = Config(
        dataset_root=dataset_root,
        output_dir=output_dir,
        max_train_samples=4000,
        max_val_samples=800,
        num_epochs=6,
        batch_size=8,
        grad_accum_steps=2,
        lr=2e-4,
        seed=42,
        use_amp=True,
        eval_every_n_epochs=2,
        modality_train="mipa",
        use_ma_module=True,
        ma_gamma=0.05,
        freeze_backbone=False,
    )
 
    experiments = []
    
    # ===================== GROUP A: rho distribution comparison =====================
    # All use: MiPa + MA (gamma=0.10) + ResNet50 backbone
    # Variable: rho_mode only
 
    # A1: Paper baseline — U(0,1)
    cfg = Config(**{**asdict(base),
        "experiment_name": "A1_rho_uniform",
        "rho_mode": "variable",
    })
    experiments.append(cfg)
 
    # A2: Beta(2,2) — bell-shaped, concentrates around 0.5
    cfg = Config(**{**asdict(base),
        "experiment_name": "A2_rho_beta_centered",
        "rho_mode": "beta_centered",
        "beta_a": 2.0,
        "beta_b": 2.0,
    })
    experiments.append(cfg)
 
    # A3: Beta(0.5,0.5) — bimodal, pushes toward extremes
    cfg = Config(**{**asdict(base),
        "experiment_name": "A3_rho_beta_bimodal",
        "rho_mode": "beta_bimodal",
        "beta_a": 0.5,
        "beta_b": 0.5,
    })
    experiments.append(cfg)
 
    # A4: Truncated Uniform U(0.15, 0.85)
    cfg = Config(**{**asdict(base),
        "experiment_name": "A4_rho_truncated_uniform",
        "rho_mode": "truncated_uniform",
    })
    experiments.append(cfg)
 
    # A5: Gaussian N(0.5, 0.15) clipped to [0,1]
    cfg = Config(**{**asdict(base),
        "experiment_name": "A5_rho_gaussian",
        "rho_mode": "gaussian",
    })
    experiments.append(cfg)
 
    return experiments
 
 
# ============================================================================
# print_results_table (corrected for new experiments)
# ============================================================================
# REPLACES the entire old print_results_table function
 
def print_results_table(all_results: List[Dict]):
    """Print a formatted results comparison table for rho distribution study."""
    print("\n" + "=" * 110)
    print("  RESULTS SUMMARY — ρ Distribution Comparison (MiPa + MA γ=0.10, ResNet50-FPN)")
    print("=" * 110)
 
    header = (
        f"{'Experiment':<28} | {'ρ Mode':<20} | {'IR AP50':>8} | {'RGB AP50':>8} | "
        f"{'AVG AP50':>8} | {'IR Prec':>7} | {'IR Rec':>6} | "
        f"{'RGB Prec':>8} | {'RGB Rec':>7} | {'Time(min)':>9}"
    )
    print(header)
    print("-" * 110)
 
    for r in all_results:
        m = r["final_metrics"]
        rho_mode = r["config"].get("rho_mode", "?")
        print(
            f"{r['experiment_name']:<28} | "
            f"{rho_mode:<20} | "
            f"{m['ir_AP50']:>8.2f} | "
            f"{m['rgb_AP50']:>8.2f} | "
            f"{m['avg_AP50']:>8.2f} | "
            f"{m['ir_precision']:>7.1f} | "
            f"{m['ir_recall']:>6.1f} | "
            f"{m['rgb_precision']:>8.1f} | "
            f"{m['rgb_recall']:>7.1f} | "
            f"{r['total_time_s']/60:>9.1f}"
        )
 
    print("-" * 110)
 
    # Find best
    best = max(all_results, key=lambda x: x["final_metrics"]["avg_AP50"])
    print(f"\n  ★ Best config: {best['experiment_name']} "
          f"(AVG AP50 = {best['final_metrics']['avg_AP50']:.2f}%)")
 
    # ---- Comparative analysis ----
    print("\n" + "=" * 110)
    print("  ANALYSIS")
    print("=" * 110)
 
    # Look up experiments by name prefix
    def find_exp(key):
        return next((r for r in all_results if key in r["experiment_name"]), None)
 
    uniform   = find_exp("A1_rho_uniform")
    beta_c    = find_exp("A2_rho_beta_centered")
    beta_bi   = find_exp("A3_rho_beta_bimodal")
    trunc     = find_exp("A4_rho_truncated")
    gaussian  = find_exp("A5_rho_gaussian")
 
    print("\n  1. ρ DISTRIBUTION IMPACT ON MODALITY BALANCE:")
    if uniform:
        baseline_avg = uniform["final_metrics"]["avg_AP50"]
        baseline_ir  = uniform["final_metrics"]["ir_AP50"]
        baseline_rgb = uniform["final_metrics"]["rgb_AP50"]
        print(f"     Paper baseline U(0,1): AVG={baseline_avg:.2f}%, "
              f"IR={baseline_ir:.2f}%, RGB={baseline_rgb:.2f}%")
 
        for label, exp in [("Beta(2,2) centered", beta_c),
                           ("Beta(0.5,0.5) bimodal", beta_bi),
                           ("Truncated U(0.15,0.85)", trunc),
                           ("Gaussian N(0.5,0.15)", gaussian)]:
            if exp:
                diff = exp["final_metrics"]["avg_AP50"] - baseline_avg
                ir_diff = exp["final_metrics"]["ir_AP50"] - baseline_ir
                rgb_diff = exp["final_metrics"]["rgb_AP50"] - baseline_rgb
                print(f"     {label:<26}: AVG {diff:+.2f}%, "
                      f"IR {ir_diff:+.2f}%, RGB {rgb_diff:+.2f}%")
 
    print("\n  2. KEY FINDINGS:")
    if uniform and beta_c:
        diff = beta_c["final_metrics"]["avg_AP50"] - uniform["final_metrics"]["avg_AP50"]
        if diff > 0:
            print(f"     → Beta(2,2) OUTPERFORMS uniform by {diff:.2f}%: "
                  f"avoiding extreme ρ values improves learning.")
        else:
            print(f"     → Beta(2,2) underperforms uniform by {abs(diff):.2f}%: "
                  f"extreme ρ values may provide useful augmentation diversity.")
 
    if uniform and beta_bi:
        diff = beta_bi["final_metrics"]["avg_AP50"] - uniform["final_metrics"]["avg_AP50"]
        if diff > 0:
            print(f"     → Bimodal Beta(0.5,0.5) OUTPERFORMS uniform by {diff:.2f}%: "
                  f"near-unimodal images help single-modality robustness.")
        else:
            print(f"     → Bimodal Beta(0.5,0.5) underperforms by {abs(diff):.2f}%: "
                  f"extreme mixing ratios hurt learning as hypothesized.")
 
    if uniform and trunc:
        diff = trunc["final_metrics"]["avg_AP50"] - uniform["final_metrics"]["avg_AP50"]
        print(f"     → Truncated uniform effect: {diff:+.2f}% vs paper baseline")
 
    print("\n  3. MODALITY SYMMETRY CHECK:")
    for label, exp in [("Uniform", uniform), ("Beta centered", beta_c),
                       ("Truncated", trunc), ("Gaussian", gaussian)]:
        if exp:
            gap = abs(exp["final_metrics"]["ir_AP50"] - exp["final_metrics"]["rgb_AP50"])
            print(f"     {label:<20}: IR-RGB gap = {gap:.2f}% "
                  f"({'balanced' if gap < 3.0 else 'IMBALANCED'})")
 
    print("\n  4. COMPARISON WITH PAPER:")
    print(f"     Paper (DINO+SWIN, full data, 12ep): MiPa+MA = 93.00 avg AP50")
    if uniform:
        print(f"     Ours  (FRCNN+R50, 4k subset, 6ep):  MiPa+MA = "
              f"{uniform['final_metrics']['avg_AP50']:.2f} avg AP50")
        print(f"     Gap expected due to: CNN backbone, subset, fewer epochs.")
 
    print("\n  5. CRITIQUE OF PAPER'S ρ ~ U(0,1) CHOICE:")
    print(f"     The paper does not justify why U(0,1) was chosen over other")
    print(f"     distributions. Our experiments test whether this is optimal.")
    if beta_c and uniform:
        if beta_c["final_metrics"]["avg_AP50"] > uniform["final_metrics"]["avg_AP50"]:
            print(f"     FINDING: U(0,1) is NOT optimal — Beta(2,2) improves results.")
            print(f"     The paper's choice allows ρ≈0 and ρ≈1 which produce")
            print(f"     near-unimodal images, wasting training diversity.")
        else:
            print(f"     FINDING: U(0,1) holds up well, suggesting the model")
            print(f"     benefits from seeing the full range of mixing ratios.")
 

def save_results_json(all_results: List[Dict], output_dir: str):
    """Save all results to JSON for later analysis."""
    # Strip non-serializable items from history
    serializable = []
    for r in all_results:
        sr = {k: v for k, v in r.items() if k != "history"}
        # Add condensed history
        if "history" in r:
            sr["train_loss_curve"] = [
                {"epoch": i + 1, "total_loss": l.get("total", 0)}
                for i, l in enumerate(r["history"]["train_losses"])
            ]
            sr["val_ir_curve"] = r["history"]["val_metrics_ir"]
            sr["val_rgb_curve"] = r["history"]["val_metrics_rgb"]
        serializable.append(sr)

    path = os.path.join(output_dir, "all_results.json")
    with open(path, "w") as f:
        json.dump(serializable, f, indent=2, default=str)
    print(f"\n  Results saved to: {path}")


def save_results_csv(all_results: List[Dict], output_dir: str):
    """Save summary results as CSV."""
    path = os.path.join(output_dir, "results_summary.csv")
    with open(path, "w") as f:
        headers = [
            "experiment", "modality_train", "rho_mode", "rho_fixed",
            "beta_a", "beta_b",
            "use_ma", "ma_gamma", "freeze_backbone", "backbone",
            "ir_AP50", "rgb_AP50", "avg_AP50",
            "ir_precision", "ir_recall", "rgb_precision", "rgb_recall",
            "time_min", "peak_gpu_gb", "best_epoch"
        ]
        f.write(",".join(headers) + "\n")

        for r in all_results:
            m = r["final_metrics"]
            c = r["config"]
            row = [
                r["experiment_name"],
                c["modality_train"],
                c["rho_mode"],
                str(c["rho_fixed_value"]),
                str(c.get("beta_a", 2.0)),
                str(c.get("beta_b", 2.0)),
                str(c["use_ma_module"]),
                str(c["ma_gamma"]),
                str(c["freeze_backbone"]),
                str(c.get("backbone", "resnet50")),
                f"{m['ir_AP50']:.2f}",
                f"{m['rgb_AP50']:.2f}",
                f"{m['avg_AP50']:.2f}",
                f"{m['ir_precision']:.1f}",
                f"{m['ir_recall']:.1f}",
                f"{m['rgb_precision']:.1f}",
                f"{m['rgb_recall']:.1f}",
                f"{r['total_time_s']/60:.1f}",
                f"{r['peak_gpu_gb']:.2f}",
                str(r["best_epoch"]),
            ]
            f.write(",".join(row) + "\n")

    print(f"  CSV saved to: {path}")


# ============================================================================
# SECTION 9: MAIN ENTRY POINT
# ============================================================================

def main():
    print("=" * 70)
    print("  MiPa: Mixed Patch Modality Agnostic Object Detection")
    print("  Reproduction & Experimental Study on LLVIP")
    print("  Target: Kaggle Free-Tier (T4 GPU)")
    print(f"  Timestamp: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 70)

    # Device setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n  Device: {device}")
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("  WARNING: No GPU detected! Training will be very slow.")

    # Dataset detection
    dataset_root = "/kaggle/input/datasets/riyajain1040/llvip-dataset/LLVIP"
    
    # Handle nested structure
    if not os.path.isdir(os.path.join(dataset_root, "visible")):
        subdirs = os.listdir(dataset_root)
        for sub in subdirs:
            sub_path = os.path.join(dataset_root, sub)
            if os.path.isdir(sub_path) and \
               os.path.isdir(os.path.join(sub_path, "visible")) and \
               os.path.isdir(os.path.join(sub_path, "infrared")):
                dataset_root = sub_path
                break
    
    print(f"\nUsing dataset root: {dataset_root}")
    if not dataset_root:
        print("\n  ERROR: LLVIP dataset not found!")
        print("  Please upload the LLVIP dataset to Kaggle with this structure:")
        print("    /kaggle/input/llvip/")
        print("      ├── visible/train/  *.jpg")
        print("      ├── visible/test/   *.jpg")
        print("      ├── infrared/train/ *.jpg")
        print("      ├── infrared/test/  *.jpg")
        print("      └── Annotations/    *.xml (VOC format)")
        print("\n  Alternatively, set dataset_root manually in the Config.")
        sys.exit(1)

    print(f"\n  Dataset root: {dataset_root}")

    # Verify structure
    for sub in ["visible/train", "visible/test", "infrared/train", "infrared/test"]:
        path = os.path.join(dataset_root, sub)
        if os.path.isdir(path):
            count = len([f for f in os.listdir(path)
                        if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            print(f"    {sub}: {count} images")

    ann_dir = find_annotation_dirs(dataset_root)
    if ann_dir:
        ann_count = len([f for f in os.listdir(ann_dir) if f.endswith(".xml")])
        print(f"    Annotations: {ann_count} XML files at {ann_dir}")
    else:
        print("    WARNING: No annotation directory found!")

    # Output directory
    output_dir = "/kaggle/working/mipa_results"
    if not os.path.isdir("/kaggle/working"):
        output_dir = "./mipa_results"
    os.makedirs(output_dir, exist_ok=True)
    print(f"\n  Output directory: {output_dir}")

    # Define experiments
    experiments = define_experiments(dataset_root, output_dir)
    print(f"\n  Total experiments: {len(experiments)}")
    for i, cfg in enumerate(experiments):
        print(f"    [{i+1}] {cfg.experiment_name}")

    # Run all experiments
    all_results = []
    total_start = time.time()

    for i, cfg in enumerate(experiments):
        try:
            result = run_experiment(cfg, device)
            all_results.append(result)

            # Save intermediate results
            save_results_json(all_results, output_dir)
            save_results_csv(all_results, output_dir)

        except Exception as e:
            print(f"\n  ERROR in experiment {cfg.experiment_name}: {e}")
            traceback.print_exc()
            all_results.append({
                "experiment_name": cfg.experiment_name,
                "error": str(e),
                "config": asdict(cfg),
                "final_metrics": {
                    "ir_AP50": 0, "rgb_AP50": 0, "avg_AP50": 0,
                    "ir_precision": 0, "ir_recall": 0,
                    "rgb_precision": 0, "rgb_recall": 0,
                    "ir_time_per_image_ms": 0, "rgb_time_per_image_ms": 0,
                },
                "best_epoch": -1,
                "total_time_s": 0,
                "peak_gpu_gb": 0,
            })

        # Force garbage collection between experiments
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

    total_time = time.time() - total_start

    # Print final results
    print_results_table(all_results)

    # Save final results
    save_results_json(all_results, output_dir)
    save_results_csv(all_results, output_dir)

    print(f"\n  Total experiment time: {total_time/60:.1f} minutes "
          f"({total_time/3600:.1f} hours)")
    print(f"  All outputs saved to: {output_dir}")
    print(f"\n  Files generated:")
    for f in sorted(os.listdir(output_dir)):
        fpath = os.path.join(output_dir, f)
        size = os.path.getsize(fpath) / 1e6
        print(f"    {f} ({size:.1f} MB)")

    print("\n" + "=" * 70)
    print("  EXPERIMENT COMPLETE")
    print("=" * 70)


if __name__ == "__main__":
    main()



  MiPa: Mixed Patch Modality Agnostic Object Detection
  Reproduction & Experimental Study on LLVIP
  Target: Kaggle Free-Tier (T4 GPU)
  Timestamp: 2026-03-28 12:19:10

  Device: cuda
  GPU: Tesla T4
  VRAM: 15.6 GB

Using dataset root: /kaggle/input/datasets/riyajain1040/llvip-dataset/LLVIP

  Dataset root: /kaggle/input/datasets/riyajain1040/llvip-dataset/LLVIP
    visible/train: 12025 images
    visible/test: 3463 images
    infrared/train: 12025 images
    infrared/test: 3463 images
    Annotations: 15488 XML files at /kaggle/input/datasets/riyajain1040/llvip-dataset/LLVIP/Annotations

  Output directory: /kaggle/working/mipa_results

  Total experiments: 5
    [1] A1_rho_uniform
    [2] A2_rho_beta_centered
    [3] A3_rho_beta_bimodal
    [4] A4_rho_truncated_uniform
    [5] A5_rho_gaussian

  EXPERIMENT: A1_rho_uniform
  Modality: mipa | rho_mode: variable | Beta(2.0,2.0) | MA: True (γ=0.05) | Backbone: resnet50
  LR: 0.0002 | BS: 8 | Epochs: 6 | Seed: 42

  Loading dataset...
 

/tmp/ipykernel_55/2711288804.py:1092: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=config.use_amp)


  [Resume] Loaded model weights from epoch 1 (optimizer reset)

  Starting training (6 epochs, resuming from epoch 1)...

  --- Epoch 2/6 ---
    [Epoch 2] Batch 1/500 | Loss: 0.2572 | Time: 2.6s | GPU: 7.28GB
    [Epoch 2] Batch 100/500 | Loss: 0.2461 | Time: 139.8s | GPU: 7.81GB
    [Epoch 2] Batch 200/500 | Loss: 0.2419 | Time: 277.4s | GPU: 7.81GB
    [Epoch 2] Batch 300/500 | Loss: 0.2385 | Time: 414.6s | GPU: 7.81GB
    [Epoch 2] Batch 400/500 | Loss: 0.2324 | Time: 552.4s | GPU: 7.81GB
    [Epoch 2] Batch 500/500 | Loss: 0.2264 | Time: 689.9s | GPU: 7.81GB
    Train Loss: 0.2264 | Time: 689.9s
    Evaluating...
    IR  AP50: 90.79% | Prec: 84.7% | Rec: 97.2%
    RGB AP50: 85.20% | Prec: 38.1% | Rec: 90.8%
    AVG AP50: 88.00%
    ✓ Best model saved (AP50_avg=88.00%)
    Peak GPU Memory: 7.81 GB

  --- Epoch 3/6 ---
    [Epoch 3] Batch 1/500 | Loss: 0.2323 | Time: 2.3s | GPU: 7.63GB
    [Epoch 3] Batch 100/500 | Loss: 0.2442 | Time: 138.6s | GPU: 7.81GB
    [Epoch 3] Batch 200/50

KeyboardInterrupt: 